# Lesson 1 — What is a System One model?

This is lesson 1 of a series building toward explaining [TypeSafe AI](https://typesafe.ai/) from fundamentals to real use cases. By the end of this lesson you'll be able to explain what makes a **System One model** different from an LLM call, and you'll run a real question against TypeSafe's `Jev` model.

Anything unclear here? Ask your agent (Claude Code) — that's what it's there for.

<details>
  <summary><strong>Primary source:</strong> docs.typesafe.ai/concepts/system-one (click to expand)</summary>

# System One

> System One models make fast, structured decisions for software. Jev is TypeSafe's flagship model and the first System One model.

System One models are a class of AI models built to make fast, structured decisions that software can use directly. A System One model evaluates a [state](https://docs.typesafe.ai/concepts/state) and returns typed answers and probabilities.

Jev is TypeSafe's flagship model and the first System One model.

Like an LLM, a System One model understands natural-language input. It returns typed decisions and probabilities rather than generated text.

> **Note:** Jev currently accepts text input only. It evaluates strings, JSON objects, and arrays of text. Images, audio, and video are not supported (yet).

## How it differs from an LLM

System One models are trained for calibrated decisions: their probabilities are optimized against outcomes to reflect uncertainty. Calibration is measured across groups of predictions; it does not guarantee that an individual answer is correct.

System One models do not write replies, produce code, or generate explanations of their reasoning. You define the possible answers through [primitives](https://docs.typesafe.ai/primitives):

| Primitive                                                    | Question                              | Example answer space                          | Example output      |
| -------------------------------------------------------------- | -------------------------------------- | --------------------------------------------- | -------------------- |
| [Choice](https://docs.typesafe.ai/primitives/choice) | Which team should handle this ticket? | `billing`, `technical`, or `account`          | `choice: "billing"` |
| [Score](https://docs.typesafe.ai/primitives/score)   | How frustrated is this customer?      | 0 = calm, 1 = frustrated, 2 = very frustrated | `score: 1.4`        |
| [Noul](https://docs.typesafe.ai/primitives/noul)     | Does this message request a refund?   | True or false                                 | `noul: 0.95`        |

These are illustrative configurations and values. The primitive pages describe the available configuration options and full response fields.

Read the [AI primer](https://docs.typesafe.ai/introduction/machine-learning-primer) to learn how System One models work and how they are trained.

> **Note:** The System One name comes from the concept Daniel Kahneman popularized in his book *Thinking, Fast and Slow*. System 1 thinking is fast and intuitive. System 2 is slower and more deliberate. Here, the emphasis is on fast, focused judgments.

## Fast judgments inside a larger workflow

For a refund request, your application can:

1. Build a state containing the customer's message, the relevant transactions, and the refund policy.
2. Ask independent questions together: whether a refund was requested, whether the evidence indicates a duplicate charge, and whether the policy supports a refund.
3. Combine the answers with deterministic checks in code, then route the case for action or review.

Once you have seen the primitives in action, you can combine them into a larger system. Because System One models return typed, constrained outputs rather than free-form text, your code can inspect and combine its answers into predictable workflows. See [How to build with TypeSafe](https://docs.typesafe.ai/concepts/how-to-build-with-system-one) for the full workflow.

Answers from System One models also include [confidence](https://docs.typesafe.ai/confidence), so you can decide when to act and when to escalate to a person or a reasoning model.

## Call a System One model

Call a System One model through one of our [client SDKs](https://docs.typesafe.ai/sdk) or `POST /v1/systemone` in the [HTTP API](https://docs.typesafe.ai/api). The `model` field selects which model handles the request. The examples in these docs use `jev-latest`, which is also the SDK default. See [Models](https://docs.typesafe.ai/models) for the available models, their prices, and their aliases.

Start with [State](https://docs.typesafe.ai/concepts/state) to prepare the input and [Primitives (Questions)](https://docs.typesafe.ai/primitives) to explore the types of questions you can ask.

</details>

## The core idea

An LLM call generates text — you then have to parse that text back into something your code can use, hoping the format holds. A **System One model** (TypeSafe's flagship is called **Jev**) skips that step: you ask it a typed question against some **state**, and it returns a typed answer directly — a chosen option, a score, or a probability. No free text, no parsing, no format drift.

TypeSafe's own docs are explicit about what this does and doesn't promise: calibration is a *population-level* statistic, not a guarantee about any single answer. Keep that distinction in your back pocket — it matters a lot more once you see the vendor's own marketing claims later in this series.

## Two things you need: `state` and a question

Every call needs:

- **`state`** — the thing being judged (a string, JSON object, or array of text). One state per request.
- One or more **questions** — each one typed. Today's type is **Choice**: pick one option from a fixed set.

Let's ask Jev to categorize a support ticket. This is TypeSafe's own canonical example (from its docs and JS SDK).

In [6]:
import { client } from "../assets/typesafe-client.ts";
import { choice } from "npm:@typesafe-ai/sdk";

const response = await client.systemOne({
  state: { document: "I found dog poo on my lawn. Please fix this ASAP." },
  questions: {
    category: choice("What is this ticket about?", {
      billing: null,
      technical: null,
      other: null,
    }),
  },
});

console.log(response.answers.category.choice);

other


## What just happened

`response.answers.category.choice` is the model's pick — one of exactly `billing`, `technical`, or `other`. There's no text to parse, and the answer literally cannot be a fourth option: the *shape* of the answer is guaranteed by construction. (Whether it picked the *correct* one is a different question — one this series comes back to.)

**Try it yourself:** change the `document` text in the cell above to something clearly technical, or something ambiguous. Before you re-run the cell, guess what `category` will come back as — then check.

## Next up

Next lesson: the other two question types (`Score` and `Noul`), and why picking the right one matters.

Questions? Ask your agent — it has the same research this lesson was built from, in `research/typesafe-ai.md`.